# Prototype Run of Markov Model for Positive and Negative Datasets of Each Structural Class

### Cell 1: Set the Working Directory

In [1]:
import os

# If notebook is inside src/, move up one directory to project root
if os.getcwd().endswith("src"):
    os.chdir("..")

print("Working directory:", os.getcwd())



Working directory: /Users/biotechiestefnie/Desktop/Algorithms_Final_Project/runs


### 2. Add src/ to Path

In [2]:
import sys

# Add src/ to Python path
sys.path.insert(0, "src")


### 3. Import Submodules From Source Directory For Implementation

In [3]:
from data_loading import load_class_seqs, read_fasta
from train_models import train_all_models
from analyze_data import (
    classify_test_set,
    compute_accuracy,
    compute_summary_stats,
    confusion_matrix,
    evaluate_model_performance
)
from output_to_file import write_all_outputs
from markov_model import (
    classify_sequence,
    log_likelihood,
    count_kmers,
    estimate_transition_probs
)



### 4. Define Prototype FASTA Paths

In [4]:
class_fasta_paths = {
    "promoter_positive": "data/prototype/promoters_positive.fa",
    "promoter_negative": "data/prototype/promoters_negative.fa",
    "exon_positive":     "data/prototype/exons_positive.fa",
    "exon_negative":     "data/prototype/exons_negative.fa",
    "intron_positive":   "data/prototype/introns_positive.fa",
    "intron_negative":   "data/prototype/introns_negative.fa",
    "repeat_positive":   "data/prototype/repeats_positive.fa",
    "repeat_negative":   "data/prototype/repeats_negative.fa"
}


### 5. Load Prototype Sequences into Model for all Classes

In [ ]:
training_data = load_class_seqs(class_fasta_paths)
len(training_data)



### 6. Prepare Test Sequences and True Class Labels

In [ ]:
# Build test set and true label mapping for prototype evaluation

test_sequences = []  # Initialize list to hold test seqs
true_labels = {}  # Initialize dict mapping seq -> true class

# Extract every seq for every class
for class_label, seq_list in training_data.items():
    for rec in seq_list:
        seq_str = rec
        test_sequences.append(seq_str)
        true_labels[seq_str] = class_label

len(test_sequences), len(true_labels)


In [ ]:
for label, seqs in training_data.items():
    print(label, type(seqs[0]))


### 7. Train Markov Models for All Classes and k Values

In [ ]:
# Train Markov models for all prototype classes over k = 1, 2, 3
# Parameter values
k_values = [1, 2, 3]  # k values under evaluation
alpha = 1  # Laplace smoothing constant

models = train_all_models(training_data, k_values, alpha)

models


### 8. Classify Each Sequence Against All Classes for Every Order k

In [ ]:
# Classify all prototype sequences for each k value

all_classification_results = {}  # Initialize dictionary for results

for k in k_values:
    # classify_test_set() returns: (sequence, predicted_class, score)
    results_k = classify_test_set(test_sequences, models, k)
    all_classification_results[k] = results_k

all_classification_results


### 9. Build Data Structure for Evaluation

In [ ]:
# Build full all_results structure required by evaluate_model_performance()

all_results = {}

for k in k_values:
    results_k = []
    raw = all_classification_results[k]   # from Cell 6

    for (seq, pred, score) in raw:
        # Placeholder class_scores: same score for all classes
        class_scores = {cls: score for cls in models[k].keys()}

        # Append full 4-tuple
        results_k.append((seq, pred, score, class_scores))

    all_results[k] = results_k

all_results



### 10a. Full Classification Analysis for a Single Order k

In [ ]:
k = 2

raw_classification_k = all_classification_results[k]

# 4-tuples for this k
results_k = all_results[k]

# Convert to dicts for per-k functions
results_dicts_k = [
    {
        "sequence": seq,
        "predicted_class": pred,
        "log_likelihood": score,
        "true_class": true_labels[seq]
    }
    for (seq, pred, score, _) in results_k
]

# Summary statistics (single k)
summary_stats_k = compute_summary_stats(results_dicts_k)

# Per-k accuracy
accuracy_k = compute_accuracy(results_dicts_k)

# Per-k confusion matrix
confusion_matrix_k = confusion_matrix(results_dicts_k)

#  Full cross-k metrics, restricted to single order k, including accuracy_vs_k, confusion_matrices, likelihood_distributions for k=2)
evaluation_k = evaluate_model_performance(
    {k: results_k},
    true_labels
)

summary_stats_k, accuracy_k, confusion_matrix_k, evaluation_k

### 10b. Compute Summary Statistics for All Orders k

In [ ]:
# Classifications per test seq for all orders k
raw_classification_all_k = all_classification_results

# Convert classify results for test seqs into dicts for all orders k
results_dicts_all_k = {
    k: [
        {
            "sequence": seq,
            "predicted_class": pred,
            "log_likelihood": score,
            "true_class": true_labels[seq]
        }
        for (seq, pred, score) in raw_classification_all_k[k]
    ]
    for k in k_values
}

# Summary statistics for all orders k
summary_stats_all_k = {
    k: compute_summary_stats(results_dicts_all_k[k])
    for k in k_values
}

# Accuracy for all orders k
accuracy_all_k = {
    k: compute_accuracy(results_dicts_all_k[k])
    for k in k_values
}

# Confusion matrices for all orders k
confusion_matrices_all_k = {
    k: confusion_matrix(results_dicts_all_k[k])
    for k in k_values
}

# Full cross-k evaluation metrics
# incl accuracy_vs_k, cross-k confusion matrices, likelihood distributions
evaluation_all_k = evaluate_model_performance(
    all_results,
    true_labels
)

# Return analysis results for all k
raw_classification_all_k, summary_stats_all_k, accuracy_all_k, confusion_matrices_all_k, evaluation_all_k


### 11. Write Results for Single k and All Orders k to Output Datafiles

In [ ]:
run = "prototype"        # this is the run name (prototype run)
base_dir = "results"

# write outputs for single k (k = 2)
run_name_single = "prototype_k2"

write_all_outputs(
    results=all_results[2],          # 4-tuples for k=2
    stats=summary_stats_k,           # summary stats for k=2
    metrics=evaluation_k,            # evaluation metrics for k=2
    dataset=run,                     # run folder name
    run_name=run_name_single,
    base_dir=base_dir
)

# write outputs for all k
run_name_all = "prototype_all_k"

# flatten all 4-tuples across k
all_results_flat = []
for k in k_values:
    all_results_flat.extend(all_results[k])

# build metrics dict in the structure write_all_outputs expects
combined_metrics_all_k = {
    "accuracy_vs_k": evaluation_all_k["accuracy_vs_k"],
    "confusion_matrices": evaluation_all_k["confusion_matrices"],
    "likelihood_distributions": evaluation_all_k["likelihood_distributions"]
}

# ---- FIX: DO NOT PASS summary_stats_all_k INTO write_all_outputs ----
# Instead, pass a dummy stats dict that satisfies write_summary_stats()
# but will be ignored because we skip summary stats for all-k.

dummy_stats = {
    "total": 0,
    "unclassified": 0,
    "neg_inf": 0,
    "class_counts": {}
}

write_all_outputs(
    results=all_results_flat,
    stats=dummy_stats,               # <-- FIXED
    metrics=combined_metrics_all_k,
    dataset=run,
    run_name=run_name_all,
    base_dir=base_dir
)

